# 01 - Data Understanding (YPerf JO28)

Objectif: comprendre la structure du dataset olympique, verifier sa qualite, et degager des insights initiaux pour orienter les predictions JO 2028.


## Plan de l'analyse

1. Charger et profiler les donnees
2. Verifier la qualite (valeurs manquantes, doublons, coherence temporelle)
3. Explorer les performances par pays, sport et genre
4. Identifier des signaux recents (momentum) vers Los Angeles 2028
5. Produire des premiers insights storytelling pour YPerf


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

sns.set_theme(style="darkgrid")


In [ ]:
DATA_PATH = Path("../data/raw/olympics_dataset.csv")
assert DATA_PATH.exists(), f"Dataset introuvable: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Years: {int(df['Year'].min())} -> {int(df['Year'].max())}")
print(f"Countries (NOC): {df['NOC'].nunique()}")
print(f"Sports: {df['Sport'].nunique()}")

df.head(3)


## 1) Data dictionary rapide

In [ ]:
data_dict = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
})

data_dict.sort_values("null_pct", ascending=False)


## 2) Qualite des donnees

In [ ]:
missing = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)
missing


In [ ]:
dup_rows = df.duplicated().sum()
print(f"Duplicate rows: {dup_rows}")

key_cols = ["player_id", "Year", "Event"]
if all(c in df.columns for c in key_cols):
    dup_key = df.duplicated(subset=key_cols).sum()
    print(f"Duplicates on {key_cols}: {dup_key}")


In [ ]:
work = df.copy()
work["Medal"] = work["Medal"].fillna("None")
work["is_medal"] = (work["Medal"] != "None").astype(int)

medal_dist = (
    work["Medal"].value_counts(dropna=False)
    .rename_axis("Medal")
    .reset_index(name="count")
)
medal_dist


## 3) Exploration metier: pays, sports, genre

In [ ]:
country_medals = (
    work.groupby("NOC", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

top_countries = country_medals.head(15)
fig = px.bar(top_countries, x="NOC", y="medals", title="Top 15 countries by historical medals")
fig.show()

top_countries


In [ ]:
sport_medals = (
    work.groupby("Sport", as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
    .sort_values("medals", ascending=False)
)

fig = px.bar(sport_medals.head(15), x="Sport", y="medals", title="Top 15 sports by medals volume")
fig.update_layout(xaxis_tickangle=-35)
fig.show()

sport_medals.head(15)


In [ ]:
sex_year = (
    work.groupby(["Year", "Sex"], as_index=False)
    .size()
    .rename(columns={"size": "entries"})
)

total_year = sex_year.groupby("Year", as_index=False)["entries"].sum().rename(columns={"entries": "total"})
sex_year = sex_year.merge(total_year, on="Year", how="left")
sex_year["share"] = sex_year["entries"] / sex_year["total"]

fig = px.line(sex_year, x="Year", y="share", color="Sex", markers=True, title="Participation share by gender over time")
fig.show()

sex_year.tail(10)


## 4) Signals recents pour JO 2028

On compare la periode recente (2016-2024) avec l'historique precedent pour detecter des progressions.


In [ ]:
recent_start = 2016

country_year = (
    work.groupby(["NOC", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

hist = country_year[country_year["Year"] < recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg_medals"})
recent = country_year[country_year["Year"] >= recent_start].groupby("NOC", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg_medals"})

momentum = hist.merge(recent, on="NOC", how="outer").fillna(0)
momentum["delta"] = momentum["recent_avg_medals"] - momentum["hist_avg_medals"]
momentum["delta_pct"] = np.where(
    momentum["hist_avg_medals"] > 0,
    momentum["delta"] / momentum["hist_avg_medals"],
    np.nan,
)

momentum_filtered = momentum[momentum["recent_avg_medals"] >= 5].sort_values("delta", ascending=False)
momentum_filtered.head(20)


In [ ]:
sport_year = (
    work.groupby(["Sport", "Year"], as_index=False)["is_medal"]
    .sum()
    .rename(columns={"is_medal": "medals"})
)

sport_hist = sport_year[sport_year["Year"] < recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "hist_avg"})
sport_recent = sport_year[sport_year["Year"] >= recent_start].groupby("Sport", as_index=False)["medals"].mean().rename(columns={"medals": "recent_avg"})

sport_momentum = sport_hist.merge(sport_recent, on="Sport", how="outer").fillna(0)
sport_momentum["delta"] = sport_momentum["recent_avg"] - sport_momentum["hist_avg"]

sport_momentum.sort_values("delta", ascending=False).head(15)


## 5) Storytelling: premiers insights YPerf

Adapter ces messages apres execution des cellules ci-dessus:

- Insight 1 (macro): quels pays dominent structurellement le classement medailles.
- Insight 2 (dynamique): quels pays montrent le plus de momentum depuis 2016.
- Insight 3 (portfolio): quels sports concentrent le volume medailles et lesquels accelerent.
- Insight 4 (equite/performance): evolution de la participation F/M et impact potentiel sur les opportunites medailles.

### Pays/sports a suivre pour 2028 (template)
- Pays etablis a surveiller: top volume + stabilite multi-editions
- Pays en progression: top `delta` recent vs historique
- Sports strategiques: top `delta` sport + adequation historique des pays cibles

### Implications ML
- Cible recommandee: nb de medailles par (NOC, Sport, Year)
- Features prioritaires: lags medailles, rolling means, volume d'entrees, ratio F, effets periode recente
- Validation: split temporel strict (train <= 2016, valid = 2020 ou 2024 selon scenario)


In [ ]:
out_dir = Path("../reports/metrics")
out_dir.mkdir(parents=True, exist_ok=True)

top_countries.to_csv(out_dir / "eda_top_countries.csv", index=False)
sport_medals.head(20).to_csv(out_dir / "eda_top_sports.csv", index=False)
momentum_filtered.head(20).to_csv(out_dir / "eda_country_momentum.csv", index=False)

print(f"EDA tables exported to: {out_dir.resolve()}")
